# 📄 Notebook 01 — PDF Extraction & Exploration

**Objectif :** Extraire et explorer le contenu textuel des rapports financiers PDF.

## Questions d'analyse
1. Quelle est la structure de chaque rapport ?
2. Combien de pages ? Combien de mots ?
3. Le texte est-il propre ou bruité ?
4. Comment découper intelligemment le texte (chunking) ?

## Pourquoi c'est important
Un système RAG est aussi bon que sa capacité à extraire
le texte correctement. Un mauvais parsing = mauvaises réponses.

In [1]:
import os
import re
import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

# Charger les variables d'environnement
load_dotenv()

# Vérifier la clé OpenAI
api_key = os.getenv("OPENAI_API_KEY")
print(f" Clé OpenAI chargée : {api_key[:20]}...")

# Dossier des rapports
REPORTS_DIR = Path('../data/reports')
reports = list(REPORTS_DIR.glob('*.pdf'))

print(f"\n {len(reports)} rapport(s) trouvé(s) :")
for r in reports:
    print(f"   → {r.name}")

 Clé OpenAI chargée : sk-proj-tXSwha83v4LS...

 2 rapport(s) trouvé(s) :
   → aN0cX55xUNkB1XBx_axa_guide_actionnaire_2023.pdf
   → totalenergies_universal-registration-document-2023_2023_en_pdf.pdf


In [2]:
def extract_pdf_content(pdf_path: Path) -> dict:
    """
    Extrait le contenu complet d'un PDF.
    Retourne un dictionnaire avec métadonnées et contenu par page.
    """
    pages_content = []
    
    with pdfplumber.open(pdf_path) as pdf:
        n_pages = len(pdf.pages)
        
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                pages_content.append({
                    'page'     : i + 1,
                    'text'     : text.strip(),
                    'n_words'  : len(text.split()),
                    'n_chars'  : len(text)
                })
    
    return {
        'filename'     : pdf_path.name,
        'n_pages'      : n_pages,
        'pages'        : pages_content,
        'total_words'  : sum(p['n_words'] for p in pages_content),
        'total_chars'  : sum(p['n_chars'] for p in pages_content),
        'pages_parsed' : len(pages_content)
    }

print(" Fonction extract_pdf_content définie")

 Fonction extract_pdf_content définie


In [ ]:
print("Extraction des rapports PDF...")
print("═" * 55)

all_docs = {}

for pdf_path in reports:
    print(f"\n📄 Traitement : {pdf_path.name}")
    doc = extract_pdf_content(pdf_path)
    all_docs[pdf_path.stem] = doc
    
    print(f"   Pages totales  : {doc['n_pages']}")
    print(f"   Pages parsées  : {doc['pages_parsed']}")
    print(f"   Mots totaux    : {doc['total_words']:,}")
    print(f"   Caractères     : {doc['total_chars']:,}")

print("\n" + "═" * 55)
print(f" {len(all_docs)} rapport(s) extrait(s)")

Extraction des rapports PDF...
═══════════════════════════════════════════════════════

📄 Traitement : aN0cX55xUNkB1XBx_axa_guide_actionnaire_2023.pdf


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names   = list(all_docs.keys())
words   = [all_docs[n]['total_words'] for n in names]
pages   = [all_docs[n]['pages_parsed'] for n in names]
colors  = ['#4f8ef7', '#3ecf8e', '#f59e0b', '#f87171'][:len(names)]

# Graphique 1 — Nombre de mots
axes[0].barh(names, words, color=colors, edgecolor='white')
axes[0].set_title('Volume de texte extrait (mots)', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
for i, v in enumerate(words):
    axes[0].text(v * 1.01, i, f'{v:,}', va='center', fontsize=9)

# Graphique 2 — Nombre de pages
axes[1].barh(names, pages, color=colors, edgecolor='white')
axes[1].set_title('Nombre de pages parsées', fontweight='bold')
axes[1].set_xlabel('Pages')
for i, v in enumerate(pages):
    axes[1].text(v * 1.01, i, str(v), va='center', fontsize=9)

plt.suptitle('Analyse des rapports financiers — Volume de contenu',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_pdf_volume.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Figure sauvegardée")

In [ ]:
# Afficher les 500 premiers caractères de chaque rapport
for name, doc in all_docs.items():
    print(f"\n{'═'*60}")
    print(f"📄 {name}")
    print(f"{'═'*60}")
    if doc['pages']:
        preview = doc['pages'][0]['text'][:500]
        print(preview)
        print(f"\n... ({doc['total_words']:,} mots au total)")

In [ ]:
fig, axes = plt.subplots(1, len(all_docs), figsize=(14, 4))

if len(all_docs) == 1:
    axes = [axes]

for ax, (name, doc) in zip(axes, all_docs.items()):
    words_per_page = [p['n_words'] for p in doc['pages']]
    ax.hist(words_per_page, bins=30, color='#4f8ef7',
            edgecolor='white', alpha=0.85)
    ax.axvline(sum(words_per_page)/len(words_per_page),
               color='red', linestyle='--', linewidth=1.5,
               label=f"Moyenne : {sum(words_per_page)//len(words_per_page)} mots")
    ax.set_title(f'{name[:30]}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Mots par page')
    ax.set_ylabel('Nombre de pages')
    ax.legend(fontsize=8)

plt.suptitle('Distribution des mots par page — Rapports financiers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_words_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def clean_text(text: str) -> str:
    """
    Nettoie le texte extrait d'un PDF.
    Supprime les artefacts courants des rapports financiers.
    """
    # Supprimer les sauts de ligne multiples
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Supprimer les espaces multiples
    text = re.sub(r' {2,}', ' ', text)
    
    # Supprimer les numéros de page isolés
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # Supprimer les caractères spéciaux parasites
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\%\€\$\/\n]', ' ', text)
    
    # Nettoyer les espaces en début/fin
    text = text.strip()
    
    return text

# Appliquer le nettoyage
for name, doc in all_docs.items():
    for page in doc['pages']:
        page['text_clean'] = clean_text(page['text'])

print(" Nettoyage appliqué sur tous les documents")

# Comparer avant / après sur un exemple
example = list(all_docs.values())[0]['pages'][2]
print(f"\n{'─'*40}")
print("AVANT nettoyage (200 chars) :")
print(example['text'][:200])
print(f"\n{'─'*40}")
print("APRÈS nettoyage (200 chars) :")
print(example['text_clean'][:200])

In [ ]:
import json
from pathlib import Path

os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder le contenu extrait et nettoyé
for name, doc in all_docs.items():
    output = {
        'filename'    : doc['filename'],
        'n_pages'     : doc['n_pages'],
        'total_words' : doc['total_words'],
        'pages'       : [
            {
                'page'       : p['page'],
                'text_clean' : p['text_clean'],
                'n_words'    : p['n_words']
            }
            for p in doc['pages']
        ]
    }
    
    out_path = Path(f'../data/processed/{name}_extracted.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f" Sauvegardé → {out_path.name}")

print(f"\n {len(all_docs)} fichier(s) JSON sauvegardé(s) dans data/processed/")

## 📋 Synthèse — Extraction PDF

### Ce qu'on a fait
1. Extraction du texte page par page avec pdfplumber
2. Calcul des statistiques de volume (mots, pages, caractères)
3. Nettoyage des artefacts PDF (espaces, numéros de page, caractères parasites)
4. Sauvegarde en JSON structuré pour la suite du pipeline

### Observations clés
- Le volume de texte varie significativement selon les rapports
- Certaines pages sont quasi-vides (tableaux, images) → à gérer dans le chunking
- Le nettoyage est essentiel avant l'embedding

### Prochaine étape → 02_chunking_embeddings.ipynb
Découper le texte en chunks intelligents et les transformer
en vecteurs sémantiques avec Sentence-Transformers.